In [140]:
%load_ext autoreload
%autoreload 2

The autoreload extension is already loaded. To reload it, use:
  %reload_ext autoreload


In [141]:
from Base_Modules.Environments import Prison
from Prison_Strategies.Basic_Strategies import *
from Base_Modules.Game_Master import Game_Master
from Base_Modules.Action import Action_History, Prison_Actions, Duel_Matrix
import pandas as pd
from collections import defaultdict

In [142]:
prison = Prison()
actions = prison.Get_Actions()

In [143]:
strategies_list = [
    Random_Strategy(actions=actions),
    Random_Strategy(actions=actions, p_coop=0.1),
    Always_Betray(actions=actions),
    Always_Cooperate(actions=actions),
    Patient_Unforgiving(actions=actions),
    Copycat(actions=actions),
    Periodic(actions=actions, period=1),
]

number_of_strategies = len(strategies_list)

In [144]:
def Get_Index_By_Name(strategies : dict[int, Strategy], name : str) -> int:
    for ix, (id, s) in enumerate(strategies.items()):
        if str(s) == name:
            return id
    for ix, (id, s) in enumerate(strategies.items()):
        if str(s).startswith(name):
            return id
    return -1

In [145]:
strategies = {}

for (i, s) in enumerate(strategies_list):
    strategies[i] = s
    s.Set_ID(i)

In [146]:
number_of_games = 1000
total_games_explicit = True
max_action_memory = -1

gm = Game_Master(prison, strategies=strategies, duel_size=2, max_action_memory=max_action_memory)
duel_matrix, rewards = gm.Tournament(number_of_games, Game_Master.Game_Type.All_Vs_All, total_games_explicit=total_games_explicit)
rewards.Sort_Total_Rewards()

In [147]:
print(duel_matrix.Get_Action_History((0, 1)).Get_Action_History())
print(Get_Index_By_Name(strategies, "Periodic"))

{0: [<Prison_Actions.Betray: 1>, <Prison_Actions.Betray: 1>, <Prison_Actions.Betray: 1>, <Prison_Actions.Cooperate: 0>, <Prison_Actions.Cooperate: 0>, <Prison_Actions.Betray: 1>, <Prison_Actions.Betray: 1>, <Prison_Actions.Cooperate: 0>, <Prison_Actions.Betray: 1>, <Prison_Actions.Cooperate: 0>, <Prison_Actions.Cooperate: 0>, <Prison_Actions.Cooperate: 0>, <Prison_Actions.Cooperate: 0>, <Prison_Actions.Cooperate: 0>, <Prison_Actions.Betray: 1>, <Prison_Actions.Cooperate: 0>, <Prison_Actions.Betray: 1>, <Prison_Actions.Betray: 1>, <Prison_Actions.Cooperate: 0>, <Prison_Actions.Cooperate: 0>, <Prison_Actions.Cooperate: 0>, <Prison_Actions.Cooperate: 0>, <Prison_Actions.Cooperate: 0>, <Prison_Actions.Cooperate: 0>, <Prison_Actions.Cooperate: 0>, <Prison_Actions.Cooperate: 0>, <Prison_Actions.Cooperate: 0>, <Prison_Actions.Betray: 1>, <Prison_Actions.Cooperate: 0>, <Prison_Actions.Betray: 1>, <Prison_Actions.Cooperate: 0>, <Prison_Actions.Cooperate: 0>, <Prison_Actions.Cooperate: 0>, <Pris

In [148]:
total_rewards = rewards.Get_Total_Rewards()
total_rewards_per_name = {str(strategies[i]):total_rewards[i] for i in total_rewards.keys()}

average_rewards_per_match = {k: (float(v)/number_of_strategies) for k, v in total_rewards_per_name.items()}
average_rewards_per_round = {k: (float(v)/(number_of_strategies*number_of_games)) for k, v in total_rewards_per_name.items()}

duel_rewards = rewards.Get_All_Duel_Rewards()
winner = next(iter(total_rewards))

print(total_rewards)

{2: 14448, 4: 14236, 1: 13602, 5: 13003, 6: 10677, 0: 10308, 3: 9351}


In [149]:
def Sort_Based_On_Total_Rewards(total_rewards, data):
    sorted_data = dict(sorted(
        data.items(),
        key=lambda kv: total_rewards.get(kv[0], 0),
        reverse=True
    ))
    return sorted_data

## Wyniki

In [150]:
average_reward_per_round_df = pd.DataFrame.from_dict(average_rewards_per_round, orient="index", columns=["Average Reward"])
average_reward_per_round_df.index.name = "Strategy Name"
average_reward_per_round_df = average_reward_per_round_df.round(3)
average_reward_per_round_df

,Average Reward
Strategy Name,
(2):Always_Betray,2.064
(4):Patient_Unforgiving (patience=1),2.034
(1):Random_Strategy (p_coop=0.1),1.943
(5):Copycat (1st:Cooperate),1.858
(6):Periodic (period=1),1.525
(0):Random_Strategy (p_coop=0.5),1.473
(3):Always_Cooperate,1.336


## Nemesis

In [151]:
from Base_Modules.Nemesis import Nemesis_Best_Enemy_Score, Nemesis_Worst_Score, Nemesis_Largest_Difference

criterion = Nemesis_Worst_Score

nemesis = criterion.Get_Nemesis(duel_rewards=duel_rewards)
nemesis = Sort_Based_On_Total_Rewards(total_rewards=total_rewards, data=nemesis)
nemesis_per_name = criterion.Translate_Nemesis_To_Strategy_Names(strategies=strategies, nemesis=nemesis)

nemesis_df = pd.DataFrame(
    [(k, v[0]) for k, v in nemesis_per_name.items()],
    columns=["Strategy", "Its nemesis"]
)

display(nemesis_df)

,Strategy,Its nemesis
0,(2):Always_Betray,(4):Patient_Unforgiving (patience=1)
1,(4):Patient_Unforgiving (patience=1),(2):Always_Betray
2,(1):Random_Strategy (p_coop=0.1),(2):Always_Betray
3,(5):Copycat (1st:Cooperate),(2):Always_Betray
4,(6):Periodic (period=1),(2):Always_Betray
5,(0):Random_Strategy (p_coop=0.5),(2):Always_Betray
6,(3):Always_Cooperate,(2):Always_Betray


In [152]:
from Base_Modules.Nemesis import Friend_Best_Total_Score

criterion = Friend_Best_Total_Score

friends = criterion.Get_Nemesis(duel_rewards=duel_rewards)
friends = Sort_Based_On_Total_Rewards(total_rewards=total_rewards, data=friends)
friends_per_name = criterion.Translate_Nemesis_To_Strategy_Names(strategies=strategies, nemesis=friends)

largets_shared_victory = 0

for (_, rewards) in friends.values():
    largets_shared_victory = max(largets_shared_victory, sum(rewards.values()))

print(f"Largest cumulated reward: {largets_shared_victory}")

friends_df = pd.DataFrame(
    [(k, v[0]) for k, v in friends_per_name.items()],
    columns=["Strategy", "Its friend"]
)

display(friends_df)

Largest cumulated reward: 6000


,Strategy,Its friend
0,(2):Always_Betray,(3):Always_Cooperate
1,(4):Patient_Unforgiving (patience=1),(3):Always_Cooperate
2,(1):Random_Strategy (p_coop=0.1),(3):Always_Cooperate
3,(5):Copycat (1st:Cooperate),(3):Always_Cooperate
4,(6):Periodic (period=1),(3):Always_Cooperate
5,(0):Random_Strategy (p_coop=0.5),(3):Always_Cooperate
6,(3):Always_Cooperate,(4):Patient_Unforgiving (patience=1)


In [153]:
# Per match

average_reward_per_match_df = pd.DataFrame.from_dict(average_rewards_per_match, orient="index", columns=["Average Reward"])
average_reward_per_match_df.index.name = "Strategy Name"
average_reward_per_match_df = average_reward_per_match_df.round(3)
# average_reward_per_match_df

In [154]:
strat_names = [str(s) for s in strategies.values()]

score_matrix = pd.DataFrame(index=strat_names, columns=strat_names, dtype=object)

for (s1, s2), scores in duel_rewards.items():
    score_matrix.loc[str(strategies[s2]), str(strategies[s1])] = (scores[s2], scores[s1])
    score_matrix.loc[str(strategies[s1]), str(strategies[s2])] = (scores[s1], scores[s2])

for s in strat_names:
    score_matrix.loc[s, s] = (0, 0)

display(score_matrix)

,(0):Random_Strategy (p_coop=0.5),(1):Random_Strategy (p_coop=0.1),(2):Always_Betray,(3):Always_Cooperate,(4):Patient_Unforgiving (patience=1),(5):Copycat (1st:Cooperate),(6):Periodic (period=1)
(0):Random_Strategy (p_coop=0.5),"(0, 0)","(808, 2898)","(508, 2968)","(3968, 1548)","(542, 2877)","(2253, 2248)","(2229, 2314)"
(1):Random_Strategy (p_coop=0.1),"(2898, 808)","(0, 0)","(882, 1472)","(4798, 303)","(913, 1363)","(1263, 1258)","(2848, 853)"
(2):Always_Betray,"(2968, 508)","(1472, 882)","(0, 0)","(5000, 0)","(1004, 999)","(1004, 999)","(3000, 500)"
(3):Always_Cooperate,"(1548, 3968)","(303, 4798)","(0, 5000)","(0, 0)","(3000, 3000)","(3000, 3000)","(1500, 4000)"
(4):Patient_Unforgiving (patience=1),"(2877, 542)","(1363, 913)","(999, 1004)","(3000, 3000)","(0, 0)","(3000, 3000)","(2997, 507)"
(5):Copycat (1st:Cooperate),"(2248, 2253)","(1258, 1263)","(999, 1004)","(3000, 3000)","(3000, 3000)","(0, 0)","(2498, 2503)"
(6):Periodic (period=1),"(2314, 2229)","(853, 2848)","(500, 3000)","(4000, 1500)","(507, 2997)","(2503, 2498)","(0, 0)"


In [155]:
sum_matrix = pd.DataFrame(index=strat_names, columns=strat_names, dtype=object)

for (s1, s2), scores in duel_rewards.items():
    sum_matrix.loc[str(strategies[s2]), str(strategies[s1])] = (scores[s2] + scores[s1])
    sum_matrix.loc[str(strategies[s1]), str(strategies[s2])] = (scores[s2] + scores[s1])

for s in strat_names:
    sum_matrix.loc[s, s] = 0

display(sum_matrix)

,(0):Random_Strategy (p_coop=0.5),(1):Random_Strategy (p_coop=0.1),(2):Always_Betray,(3):Always_Cooperate,(4):Patient_Unforgiving (patience=1),(5):Copycat (1st:Cooperate),(6):Periodic (period=1)
(0):Random_Strategy (p_coop=0.5),0,3706,3476,5516,3419,4501,4543
(1):Random_Strategy (p_coop=0.1),3706,0,2354,5101,2276,2521,3701
(2):Always_Betray,3476,2354,0,5000,2003,2003,3500
(3):Always_Cooperate,5516,5101,5000,0,6000,6000,5500
(4):Patient_Unforgiving (patience=1),3419,2276,2003,6000,0,6000,3504
(5):Copycat (1st:Cooperate),4501,2521,2003,6000,6000,0,5001
(6):Periodic (period=1),4543,3701,3500,5500,3504,5001,0


In [156]:
victory_matrix = score_matrix.apply(lambda col: col.map(lambda x: int(x[0] > x[1])))
for s in strat_names:
    victory_matrix.loc[s, s] = float("NaN")

In [157]:
def color_cell(x):
    if not isinstance(x, tuple):
        return ""
    if x[0] == 0 and x[1] == 0:
        return "background-color: black"
    elif x[0] > x[1]:
        return "background-color: green"
    elif x[0] < x[1]:
        return "background-color: darkred"
    else:
        return "background-color: gray"

styled_score_matrix = (
    score_matrix.style
    .map(color_cell)
    .set_properties(**{
        "border": "1px solid black"
    })
    .set_table_styles([
        {"selector": "th", "props": [("border", "2px solid black")]},
        {"selector": "td", "props": [("border", "2px solid black")]}
    ])
)

display(styled_score_matrix)

,(0):Random_Strategy (p_coop=0.5),(1):Random_Strategy (p_coop=0.1),(2):Always_Betray,(3):Always_Cooperate,(4):Patient_Unforgiving (patience=1),(5):Copycat (1st:Cooperate),(6):Periodic (period=1)
(0):Random_Strategy (p_coop=0.5),"(0, 0)","(808, 2898)","(508, 2968)","(3968, 1548)","(542, 2877)","(2253, 2248)","(2229, 2314)"
(1):Random_Strategy (p_coop=0.1),"(2898, 808)","(0, 0)","(882, 1472)","(4798, 303)","(913, 1363)","(1263, 1258)","(2848, 853)"
(2):Always_Betray,"(2968, 508)","(1472, 882)","(0, 0)","(5000, 0)","(1004, 999)","(1004, 999)","(3000, 500)"
(3):Always_Cooperate,"(1548, 3968)","(303, 4798)","(0, 5000)","(0, 0)","(3000, 3000)","(3000, 3000)","(1500, 4000)"
(4):Patient_Unforgiving (patience=1),"(2877, 542)","(1363, 913)","(999, 1004)","(3000, 3000)","(0, 0)","(3000, 3000)","(2997, 507)"
(5):Copycat (1st:Cooperate),"(2248, 2253)","(1258, 1263)","(999, 1004)","(3000, 3000)","(3000, 3000)","(0, 0)","(2498, 2503)"
(6):Periodic (period=1),"(2314, 2229)","(853, 2848)","(500, 3000)","(4000, 1500)","(507, 2997)","(2503, 2498)","(0, 0)"


In [158]:
import webbrowser
store_styled_matrix_in_html = False
if store_styled_matrix_in_html:
    styled_score_matrix.to_html("styled_score_matrix.html")
    webbrowser.open_new_tab("styled_score_matrix.html")